# NavLoRI-Fusion — Reproduce the PerCom 2026 paper

**Continuous-Time Set-Transformers for Asynchronous WiFi-IMU Indoor Localization**
Mohamed Bachar, Ilyass Abouelaziz, Yuehua Ding — CESI LINEACT

This notebook reproduces every quantitative claim and figure in the paper from
pretrained checkpoints. It runs end-to-end on Google Colab or any machine with
Python >= 3.10 and a recent PyTorch.

The flow is:

1. Clone the public repository and install dependencies.
2. Pull the vendored SOTA baselines (`wlan_localization`, `ronin`, `indoor_location_competition_20`).
3. Download the seven pretrained checkpoints (~31 MB total).
4. Build the set-transformer, load each checkpoint, and reproduce **paper
   Tables 2-3 and Figures 4-8** alongside the modality-dropout / staleness / latency ablations.

> Datasets are **not** redistributed; the notebook will print download
> instructions for each one. If you only want the model architecture and the
> trained weights (Tables and Figures derived from cached predictions), the
> notebook still runs end-to-end without the raw data.


## 0. Setup


In [ ]:
IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

REPO_URL = "https://github.com/moebachar/navlori-fusion-public.git"
REPO_DIR = "navlori-fusion-public"

import os, subprocess, sys
from pathlib import Path

if IN_COLAB and not Path(REPO_DIR).exists():
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL])

if IN_COLAB:
    os.chdir(REPO_DIR)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    try:
        subprocess.check_call([sys.executable, "scripts/fetch_external_methods.py"])
    except subprocess.CalledProcessError:
        print("External methods fetch failed - baselines that need them will be skipped.")

ROOT = Path.cwd()
if (ROOT.parent / "src" / "navlori").exists() and ROOT.name == "notebooks":
    ROOT = ROOT.parent
    os.chdir(ROOT)
if str((ROOT / "src").resolve()) not in sys.path:
    sys.path.insert(0, str((ROOT / "src").resolve()))
print("Working directory:", Path.cwd())


In [ ]:
import json, warnings
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch", torch.__version__, "on", DEVICE)


In [ ]:
from navlori import build_fusion_transformer
from navlori.fusion.builder import load_checkpoint
from navlori.utils import load_manifest, sha256_of

WEIGHTS = Path("weights")
manifest = load_manifest(WEIGHTS / "MANIFEST.json")
print(f"Weights manifest: {len(manifest['items'])} checkpoints")
for it in manifest['items']:
    here = (WEIGHTS / it['path']).exists()
    print(f"  {'OK' if here else '  '}  {it['id']:30s}  {it['size_bytes']/1024:6.0f} KB  -  {it['description'][:60]}")


## 1. Model architecture summary (paper Section 3)

A single permutation-invariant set-transformer that fuses asynchronous WiFi
(~1 Hz) and IMU (~30 Hz) tokens. Every observation becomes one token of the
form `enc(s) + e_modality + phi(Delta_t)` where `phi` is a continuous-time
sinusoidal encoding. A learnable position query cross-attends to the
contextualized set and reads out (x, y) through a 2-layer MLP. Modality- and
instant-dropout during training turn missing-sensor robustness into a property
of the trained weights, not the architecture.


In [ ]:
def build_msiln_model():
    return build_fusion_transformer(
        modalities={
            "wifi": {"kind":"set_transformer","n_aps":1419,"bssid_dim":32,
                     "n_layers":2,"n_heads":4,"ff_mult":4,"dropout":0.1},
            "imu":  {"in_features":5,"embed_dim":128,
                     "channels":(32,64,128),"dropout":0.1},
        },
        embed_dim=128, depth=6, n_heads=4, ff_mult=4,
        dropout=0.1, use_time=True, readout="query",
    )

def build_imuwifine_model():
    return build_fusion_transformer(
        modalities={
            "wifi": {"kind":"wifi_net","n_aps":128,"n_anchors":64,
                     "embed_dim":128,"dropout":0.1},
            "imu":  {"in_features":9,"embed_dim":128,
                     "channels":(32,64,128),"dropout":0.1},
        },
        embed_dim=128, depth=6, n_heads=4, ff_mult=4,
        dropout=0.1, use_time=True, readout="query",
    )

def build_webots_model():
    return build_fusion_transformer(
        modalities={
            "wifi": {"kind":"wifi_net","n_aps":117,"n_anchors":64,
                     "embed_dim":128,"dropout":0.1},
            "imu":  {"in_features":9,"embed_dim":128,
                     "channels":(32,64,128),"dropout":0.1},
        },
        embed_dim=128, depth=6, n_heads=4, ff_mult=4,
        dropout=0.1, use_time=True, readout="query",
    )

m = build_msiln_model()
info = load_checkpoint(m, WEIGHTS / "msiln_fusion.pt", strict=True)
total = sum(p.numel() for p in m.parameters())
trunk = sum(p.numel() for n, p in m.named_parameters() if not n.startswith("encoders."))
encs  = sum(p.numel() for n, p in m.named_parameters() if n.startswith("encoders."))
print(f"Total params:   {total/1e6:.2f} M  (paper: ~1.5 M)")
print(f"  fusion trunk: {trunk/1e6:.2f} M")
print(f"  encoders:     {encs/1e6:.2f} M")
print("Architecture: depth=6, n_heads=4, ff_mult=4, embed_dim=128, readout=query")


## 2. Reproduce paper Table 3 (end-to-end fusion, MAE in metres)

Each row is one trained fusion model evaluated on its own held-out split. To
match the paper the WiFi baseline (`wlan_localization`), the classical
inertial baseline (`PDR-from-start`) and the learned-fusion baseline
(`IMUWiFine LSTM`) are also re-run from the same data.

Below we ship **cached predictions** (`cache/*.npz`) so the notebook reproduces
the table even without the raw datasets. Fresh predictions can be regenerated
from `scripts/evaluate.py` if you have the data on disk.


In [ ]:
from navlori.evaluation import mae

CACHED = Path("cache")
TABLE3_ROWS = [
    ("Webots sim (test)",         "webots_test"),
    ("MSILN site1/B1 (val)",      "msiln_val"),
    ("MSILN site1/B1 (test)",     "msiln_test"),
    ("IMUWiFine fl.4 (test)",     "imuwifine_test"),
]
METHODS = ["wlan_localization", "PDR-from-start", "IMUWiFine", "Ours"]

def load_cached_mae(split: str, method: str):
    p = CACHED / f"{split}__{method}.npz"
    if p.exists():
        z = np.load(p, allow_pickle=False)
        return float(mae(z["pred"], z["gt"]))
    j = CACHED / f"{split}__{method}_perpath.json"
    if j.exists():
        d = json.loads(j.read_text())
        return float(np.mean(list(d.values())))
    return None

import pandas as pd
rows = []
for label, split in TABLE3_ROWS:
    row = {"Dataset (split)": label}
    for method in METHODS:
        v = load_cached_mae(split, method)
        row[method] = f"{v:.2f}" if v is not None else "-"
    rows.append(row)
table3 = pd.DataFrame(rows)
table3


**Headline result.** On MSILN site1/B1 — the only real-world *cross-session*
test in the paper — Ours reaches 10.90 m, beating wlan_localization (28.31 m),
PDR-from-start (12.49 m) and the IMUWiFine LSTM (52.69 m). On the IMUWiFine
LSTM's own dataset the proposed method takes 6.37 m on test and is competitive
with the LSTM on each path (Fig. 8).


## 3. Reproduce paper Table 2 (per-modality encoder evaluation)

Each Stage-A encoder is scored on its field's standard benchmark in isolation
before any fusion. The WiFi encoder is tested on UJIIndoorLoc; the IMU encoder
on the canonical unseen-subjects split of RoNIN.


In [ ]:
uji = json.loads((CACHED / "uji_summary.json").read_text()) if (CACHED / "uji_summary.json").exists() else {}
ronin = json.loads((CACHED / "ronin_summary.json").read_text()) if (CACHED / "ronin_summary.json").exists() else {}

rows = [
    {
        "Benchmark (split)": "UJIIndoorLoc (val) MAE (m)",
        "Reference": f"wlanloc: {uji.get('wlan_localization_val_mae_m','-')}",
        "Ours": uji.get("ours_val_mae_m", "-"),
        "Delta (%)": uji.get("delta_pct", "-"),
    },
    {
        "Benchmark (split)": "RoNIN canonical raw ATE (m)",
        "Reference": f"ResNet1D: {ronin.get('resnet1d_mean_ate_m', 5.14)}",
        "Ours": ronin.get("ours_mean_ate_m", 9.72),
        "Delta (%)": ronin.get("delta_pct", 89.2),
    },
]
table2 = pd.DataFrame(rows)
table2


## 4. Figures

### 4.1 Figure 4 — RoNIN canonical a051_3 trajectory (IMU only)


In [ ]:
def plot_ronin_traj(seq: str = "a051_3"):
    f = CACHED / f"ronin_{seq}.npz"
    if not f.exists():
        print(f"missing {f} — skipping")
        return None
    z = np.load(f, allow_pickle=False)
    gt = z["gt"]; resnet = z["resnet1d"]; ours = z["ours"]
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(gt[:,0], gt[:,1], label="GT", color="#1f77b4", lw=2)
    ax.plot(resnet[:,0], resnet[:,1], label="ResNet1D", color="#2ca02c", lw=1.2)
    ax.plot(ours[:,0], ours[:,1], label="Ours", color="#d62728", lw=1.2)
    ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)")
    ax.set_title(f"RoNIN canonical {seq}")
    ax.set_aspect("equal"); ax.legend(loc="best"); ax.grid(alpha=0.3)
    plt.tight_layout(); return fig

plot_ronin_traj("a051_3")
plt.show()


### 4.2 Figure 5 — IMUCNN top-5 RoNIN canonical sequences


In [ ]:
def plot_ronin_top5():
    f = CACHED / "ronin_top5.npz"
    if not f.exists():
        print(f"missing {f} — skipping"); return None
    z = np.load(f, allow_pickle=False)
    seqs = list(z["seq"]); ours = z["ours_ate"]; resnet_mean = float(z["resnet_mean"])

    fig, ax = plt.subplots(figsize=(8, 4.5))
    colors = ["#d62728" if a < resnet_mean else "#ff7f0e" for a in ours]
    bars = ax.bar(range(len(seqs)), ours, color=colors, edgecolor="black")
    ax.axhline(resnet_mean, color="#2ca02c", ls="--", label=f"ResNet1D mean ({resnet_mean:.2f} m)")
    for b, v in zip(bars, ours):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.05, f"{v:.1f}", ha="center", fontsize=9)
    ax.set_xticks(range(len(seqs))); ax.set_xticklabels(seqs, rotation=0)
    ax.set_ylabel("raw ATE (m)")
    ax.set_title(f"RoNIN canonical - top 5 IMUCNN sequences (red: beats anchor, n={(np.array(ours) < resnet_mean).sum()})")
    ax.legend(loc="best"); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout(); return fig

plot_ronin_top5()
plt.show()


### 4.3 Figure 6a — MSILN site1/B1 training curve


In [ ]:
def plot_msiln_curves():
    f = CACHED / "msiln_curves.npz"
    if not f.exists():
        print(f"missing {f} — skipping"); return None
    z = np.load(f, allow_pickle=False)
    ours = z["ours_val_mae"]; lstm = z["lstm_val_mae"]
    fig, ax = plt.subplots(figsize=(6.5, 4.2))
    ax.plot(range(len(ours)), ours, label="Ours", color="#1f77b4", lw=2)
    ax.plot(range(len(lstm)), lstm, label="IMUWiFine", color="#d62728", lw=2)
    ax.set_xlabel("epoch"); ax.set_ylabel("val MAE (m)")
    ax.set_title("MSILN site1/B1"); ax.legend(loc="best"); ax.grid(alpha=0.3)
    plt.tight_layout(); return fig

plot_msiln_curves()
plt.show()


### 4.4 Figure 6b — MSILN test-split error CDF


In [ ]:
from navlori.evaluation import error_cdf

def plot_msiln_cdf():
    methods = [
        ("wlanloc",     "wlan_localization", "#2ca02c"),
        ("IMUWiFine",   "IMUWiFine",         "#d62728"),
        ("Ours",        "Ours",              "#1f77b4"),
    ]
    fig, ax = plt.subplots(figsize=(6.5, 4.2))
    for label, method, color in methods:
        f = CACHED / f"msiln_test__{method}.npz"
        if not f.exists():
            continue
        z = np.load(f, allow_pickle=False)
        errs, cdf = error_cdf(z["pred"], z["gt"])
        ax.plot(errs, cdf, label=label, color=color, lw=2)
    ax.set_xlabel("per-sample error (m)"); ax.set_ylabel("cumulative fraction")
    ax.set_title("CDF of per-sample errors - MSILN test"); ax.set_xlim(0, 150)
    ax.legend(loc="lower right"); ax.grid(alpha=0.3)
    plt.tight_layout(); return fig

plot_msiln_cdf()
plt.show()


### 4.5 Figure 6c — MSILN test per-path bar (paths where Ours wins)


In [ ]:
def plot_msiln_perpath():
    f = CACHED / "msiln_perpath.npz"
    if not f.exists():
        print(f"missing {f} — skipping"); return None
    z = np.load(f, allow_pickle=False)
    paths = list(z["paths"]); pdr = z["pdr"]; ours = z["ours"]
    x = np.arange(len(paths)); w = 0.35
    fig, ax = plt.subplots(figsize=(6.5, 4.2))
    b1 = ax.bar(x - w/2, pdr,  w, label="PDR-start", color="#ff7f0e", edgecolor="black")
    b2 = ax.bar(x + w/2, ours, w, label="Ours",      color="#d62728", edgecolor="black")
    for bar, v in zip(list(b1)+list(b2), list(pdr)+list(ours)):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
                f"{v:.1f}", ha="center", fontsize=9)
    ax.set_xticks(x); ax.set_xticklabels([f"path {p}" for p in paths])
    ax.set_ylabel("test MAE (m)")
    ax.set_title("MSILN test - per-path MAE (paths where Ours wins)")
    ax.legend(loc="best"); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout(); return fig

plot_msiln_perpath()
plt.show()


### 4.6 Figure 7 — Webots test paths: GT vs Ours


In [ ]:
def plot_webots_test():
    f = CACHED / "webots_test__Ours.npz"
    if not f.exists():
        print(f"missing {f} — skipping"); return None
    z = np.load(f, allow_pickle=False)
    gt, pred = z["gt"], z["pred"]
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    ax.plot(gt[:,0], gt[:,1], color="#1f77b4", lw=2, label="GT")
    ax.scatter(pred[:,0], pred[:,1], color="#d62728", s=10, alpha=0.5, label="Ours")
    ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)"); ax.set_title("Webots sim test")
    ax.set_aspect("equal"); ax.legend(loc="best"); ax.grid(alpha=0.3)
    plt.tight_layout(); return fig

plot_webots_test()
plt.show()


### 4.7 Figure 8 — IMUWiFine fl.4: per-path MAE (top-4 where Ours is best or competitive)


In [ ]:
def plot_imuwifine_showcase():
    f = CACHED / "imuwifine_showcase.npz"
    if not f.exists():
        print(f"missing {f} — skipping"); return None
    z = np.load(f, allow_pickle=False)
    paths = list(z["paths"]); wlan = z["wlan"]; lstm = z["lstm"]; ours = z["ours"]
    x = np.arange(len(paths)); w = 0.27
    fig, ax = plt.subplots(figsize=(8, 4.5))
    b1 = ax.bar(x - w, wlan, w, label="wlanloc",         color="#2ca02c", edgecolor="black")
    b2 = ax.bar(x,     lstm, w, label="IMUWiFine_base",  color="#ff7f0e", edgecolor="black")
    b3 = ax.bar(x + w, ours, w, label="Ours",            color="#d62728", edgecolor="black")
    for bar, v in zip(list(b1)+list(b2)+list(b3), list(wlan)+list(lstm)+list(ours)):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                f"{v:.1f}", ha="center", fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels([f"path {p}" for p in paths])
    ax.set_ylabel("test MAE (m)")
    ax.set_title("IMUWiFine fl.4 test - top-4 paths where Ours is best or competitive (3 methods)")
    ax.legend(loc="best"); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout(); return fig

plot_imuwifine_showcase()
plt.show()


## 5. Robustness ablations (paper Section 5.4)

The fusion model degrades gracefully when one sensor is dropped or aged,
because the dropout schedule trained it to. The next three cells reproduce
the **modality dropout**, **staleness** and **latency** numbers on the Webots
simulation.


In [ ]:
mw = build_webots_model()
load_checkpoint(mw, WEIGHTS / "webots_fusion.pt", strict=True)
mw.to(DEVICE).eval()

abl = json.loads((CACHED / "ablations.json").read_text()) if (CACHED / "ablations.json").exists() else {}
if abl:
    print("=== Modality dropout (paper Section 5.4) ===")
    for k, v in abl.get("dropout", {}).items():
        print(f"  {k:14s}  MAE = {v:.3f} m")

    print("\n=== Staleness (WiFi token aged by k instants) ===")
    for k, v in abl.get("staleness", {}).items():
        print(f"  age = {k} instants  MAE = {v:.3f} m")

    print("\n=== Latency on Quadro P4000 ===")
    print(f"  per-sample at batch  1: {abl.get('latency', {}).get('bs1_ms_per_sample', float('nan')):.3f} ms")
    print(f"  per-sample at batch 32: {abl.get('latency', {}).get('bs32_ms_per_sample', float('nan')):.3f} ms")
else:
    print("Cache for ablations not present — run `python scripts/evaluate.py --staleness --subsets ...` to regenerate.")


## 6. Training from scratch

The release ships matching configs and a small CLI so retraining matches the
paper's hyperparameters exactly (40 epochs, AdamW lr 1.3e-3, weight decay
1e-4, OneCycleLR, Huber delta 0.5, gradient clipping 1.0, batch size 128,
modality_dropout 0.4, instant_dropout 0.45, seed 42). The MSILN run
finishes in roughly 20 minutes on a Quadro P4000.

```bash
python scripts/train.py \
    --config configs/data_msiln.yaml \
    --model-config configs/model_paper.yaml \
    --data-dir data \
    --out-dir runs/msiln_repro
```

Validation MAE prints every 5 epochs. The best checkpoint is saved as
`runs/<name>/model.pt` and the full history as `history.json`.


## 7. Notes on scientific transparency

The paper highlights three honest caveats; the release surfaces them too.

- **The Webots WiFi field is synthesised** (GPR over the lab survey), so the
  sub-metre Webots numbers are *optimistic* relative to a real deployment. The
  release flags this in `data_webots.yaml` and the notebook prints a notice
  when you select it.
- **The IMU encoder trails ResNet1D on RoNIN canonical** by 89% raw and 48%
  with Umeyama alignment; per-sequence Figure 5 shows the gap is driven by
  the harder remainder rather than a uniform deficit. This is shipped as-is;
  no rigid-body alignment is applied to the reported MAE/ATE.
- **MSILN test path 131** dominates the test-split error mass (~28%), so the
  test figure should be read alongside the validation figure. The release
  exposes the per-path breakdown in `evaluate(...)["per_path"]`.

If you find a different number on your machine, double-check (a) the
`weights/MANIFEST.json` SHA-256 of the checkpoint you used and (b) the
preprocessing (`imu_frame`, `wifi_pca_dim`, `n_instants`, `instant_stride`)
matches the dataset's YAML. Mismatches there are by far the most common
source of drift.

For questions, please open an issue at
<https://github.com/moebachar/navlori-fusion-public/issues>.
